# Assignment: Real-Time Music Analytics

This notebook is the cleaned submission version.

It performs real-time analytics on Kafka music events, joins the streaming events with the static MusicStore PostgreSQL data, and writes the required output tables into the `streaming_data` schema.

## Assignment coverage checklist

| Requirement from `assignment (5).ipynb` | Completed in this notebook |
|---|---|
| Configure Spark for Kafka and PostgreSQL/JDBC | Yes |
| Read real-time music events from Kafka | Yes |
| Define schema for incoming Kafka JSON | Yes |
| Join Kafka events with static PostgreSQL track/genre/album/artist data | Yes |
| Task 1: Top 20 playlists for Pop, Jazz, Rock | Yes |
| Task 2: Top 20 playlists for male and female listeners | Yes |
| Task 2 extra: one gender + genre playlist | Yes, male + Rock |
| Task 3: Top 20 albums from played tracks | Yes |
| Save result tables to PostgreSQL schema `streaming_data` | Yes |


## 1. Imports

Only imports are placed here. Spark is started in the next section after the configuration is defined.

In [1]:
import os
import re
import shutil
import time

from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    IntegerType,
)

## 2. Configuration

This section contains the PostgreSQL, Kafka, and table-name settings.  

In [ ]:
# Student / output table prefix
STUDENT_PREFIX = "ds25m028"

# Kafka configuration
KAFKA_BOOTSTRAP_SERVERS = "46.225.20.89:9092"
KAFKA_TOPIC = "music-fhtw"
KAFKA_STARTING_OFFSETS = "latest"

# PostgreSQL configuration
POSTGRES_HOST = "fhtw-big-data.postgres.database.azure.com"
POSTGRES_PORT = "5432"
POSTGRES_DB = "music_store"
POSTGRES_USER = "student"
POSTGRES_PASSWORD = "reRZ2pjg1WxqlwjU"

# Static MusicStore tables are in public.
SOURCE_SCHEMA = "public"

# Assignment output tables must be written to streaming_data.
OUTPUT_SCHEMA = "streaming_data"

# Azure PostgreSQL requires SSL.
POSTGRES_URL = (
    f"jdbc:postgresql://{POSTGRES_HOST}:{POSTGRES_PORT}/{POSTGRES_DB}"
    "?sslmode=require"
)

# How long each streaming query should run before it is stopped.
PROCESSING_SECONDS = 30

# Local checkpoint base directory.
CHECKPOINT_BASE = f"/tmp/{STUDENT_PREFIX}_assignment3_checkpoints"

print("POSTGRES_URL:", POSTGRES_URL)
print("Kafka topic:", KAFKA_TOPIC)
print("Output schema:", OUTPUT_SCHEMA)
print("Student prefix:", STUDENT_PREFIX)

POSTGRES_URL: jdbc:postgresql://fhtw-big-data.postgres.database.azure.com:5432/music_store?sslmode=require
Kafka topic: music-fhtw
Output schema: streaming_data
Student prefix: ds25m028


## 3. Start Spark

Run this cell once after restarting the kernel.

The PostgreSQL JDBC driver and Spark Kafka connector are loaded through Maven packages. This avoids the earlier `ClassNotFoundException: org.postgresql.Driver` problem.

In [3]:
spark = (
    SparkSession.builder
    .appName("Assignment3_RealTime_Music_Analytics")
    .master("local[*]")
    .config(
        "spark.jars.packages",
        "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.1,org.postgresql:postgresql:42.7.3",
    )
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("Spark version:", spark.version)
print("Packages:", spark.sparkContext.getConf().get("spark.jars.packages"))

26/06/22 22:28:16 WARN Utils: Your hostname, codespaces-f5202d resolves to a loopback address: 127.0.0.1; using 10.0.11.180 instead (on interface eth0)
26/06/22 22:28:16 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/usr/local/sdkman/candidates/spark/3.5.1/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/vscode/.ivy2/cache
The jars for the packages stored in: /home/vscode/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
org.postgresql#postgresql added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-589456c9-ea58-441e-b986-86c9125ca30f;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.1 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.1 in central
	found org.apache.kafka#kafka-clients;3.4.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.3 in central
	found org.slf4j#slf4j-api;2.0.7 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.commons#commons-pool2;2.11.1 in central
	found org.pos

Spark version: 3.5.1
Packages: org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.1,org.postgresql:postgresql:42.7.3


## 4. Helper functions

These functions keep the notebook readable and avoid repeating long JDBC read/write code.

In [5]:
jdbc_props = {
    "user": POSTGRES_USER,
    "password": POSTGRES_PASSWORD,
    "driver": "org.postgresql.Driver",
}


def read_jdbc_query(sql_query: str) -> DataFrame:
    """
    Read the result of a SQL query from PostgreSQL.

    Spark JDBC requires a custom SQL query to be wrapped in parentheses
    and given an alias.
    """
    return spark.read.jdbc(
        url=POSTGRES_URL,
        table=f"({sql_query}) AS q",
        properties=jdbc_props,
    )


def read_public_table(table_name: str) -> DataFrame:
    """
    Read one complete table from the public schema.

    This direct subquery format was tested successfully for the MusicStore tables.
    """
    return spark.read.jdbc(
        url=POSTGRES_URL,
        table=f"(SELECT * FROM {SOURCE_SCHEMA}.{table_name}) AS t",
        properties=jdbc_props,
    )


def write_output_table(df: DataFrame, table_name: str, mode: str = "overwrite") -> None:
    """
    Write a Spark DataFrame to the assignment output schema in PostgreSQL.
    """
    (
        df.write
        .mode(mode)
        .jdbc(
            url=POSTGRES_URL,
            table=f"{OUTPUT_SCHEMA}.{table_name}",
            properties=jdbc_props,
        )
    )


def prepare_checkpoint(name: str) -> str:
    """
    Remove and recreate a checkpoint path so the notebook can be rerun cleanly.
    """
    path = f"{CHECKPOINT_BASE}/{name}"
    shutil.rmtree(path, ignore_errors=True)
    return path


def camel_to_snake(name: str) -> str:
    """
    Convert column names such as TrackId, trackId, or First Name to lower snake_case.
    """
    name = re.sub(r"(.)([A-Z][a-z]+)", r"\1_\2", name)
    name = re.sub(r"([a-z0-9])([A-Z])", r"\1_\2", name)
    return name.lower().replace(" ", "_")


def normalize_columns(df: DataFrame) -> DataFrame:
    """
    Normalize all DataFrame column names to lower snake_case.
    """
    for old_name in df.columns:
        df = df.withColumnRenamed(old_name, camel_to_snake(old_name))
    return df

## 5. Load static PostgreSQL data

The assignment asks us to join streaming events with static MusicStore data.  
The relevant static tables are:

- `tracks`
- `genres`
- `albums`
- `artists`
- `customers`

In [6]:
tracks_raw = normalize_columns(read_public_table("tracks"))
genres_raw = normalize_columns(read_public_table("genres"))
albums_raw = normalize_columns(read_public_table("albums"))
artists_raw = normalize_columns(read_public_table("artists"))
customers_raw = normalize_columns(read_public_table("customers"))

print("Tracks:")
tracks_raw.printSchema()

print("Genres:")
genres_raw.printSchema()

print("Albums:")
albums_raw.printSchema()

print("Artists:")
artists_raw.printSchema()

print("Customers:")
customers_raw.printSchema()

Tracks:
root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- album_id: integer (nullable = true)
 |-- media_type_id: integer (nullable = true)
 |-- genre_id: integer (nullable = true)
 |-- composer: string (nullable = true)
 |-- milliseconds: integer (nullable = true)
 |-- bytes: integer (nullable = true)
 |-- unit_price: decimal(10,2) (nullable = true)

Genres:
root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)

Albums:
root
 |-- id: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- artist_id: integer (nullable = true)

Artists:
root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)

Customers:
root
 |-- id: integer (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- company: string (nullable = true)
 |-- address: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- country: string (nullable 

## 6. Prepare track metadata

The source database uses generic `id` and `name` columns in several tables.  
We rename them before joining so the final metadata is clear.

In [7]:
tracks = (
    tracks_raw
    .withColumnRenamed("id", "track_id")
    .withColumnRenamed("name", "track_name")
)

genres = (
    genres_raw
    .withColumnRenamed("id", "genre_id")
    .withColumnRenamed("name", "genre_name")
)

albums = (
    albums_raw
    .withColumnRenamed("id", "album_id")
    .withColumnRenamed("title", "album_name")
)

artists = (
    artists_raw
    .withColumnRenamed("id", "artist_id")
    .withColumnRenamed("name", "artist_name")
)

customers = (
    customers_raw
    .withColumnRenamed("id", "customer_id")
    .withColumnRenamed("first_name", "customer_first_name")
    .withColumnRenamed("last_name", "customer_last_name")
)

track_metadata = (
    tracks
    .join(genres, on="genre_id", how="left")
    .join(albums, on="album_id", how="left")
    .join(artists, on="artist_id", how="left")
    .select(
        "track_id",
        "track_name",
        "genre_id",
        "genre_name",
        "album_id",
        "album_name",
        "artist_id",
        "artist_name",
    )
)

track_metadata.show(10, truncate=False)
track_metadata.printSchema()

+--------+---------------------------------------+--------+----------+--------+-------------------------------------+---------+-----------+
|track_id|track_name                             |genre_id|genre_name|album_id|album_name                           |artist_id|artist_name|
+--------+---------------------------------------+--------+----------+--------+-------------------------------------+---------+-----------+
|1       |For Those About To Rock (We Salute You)|1       |Rock      |1       |For Those About To Rock We Salute You|1        |AC/DC      |
|2       |Balls to the Wall                      |1       |Rock      |2       |Balls to the Wall                    |2        |Accept     |
|3       |Fast As a Shark                        |1       |Rock      |3       |Restless and Wild                    |2        |Accept     |
|4       |Restless and Wild                      |1       |Rock      |3       |Restless and Wild                    |2        |Accept     |
|5       |Princess o

## 7. Read Kafka stream

Kafka provides the real-time music events.  
The `value` field contains the JSON event, so we cast it from binary to string.

In [8]:
kafka_raw = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP_SERVERS)
    .option("subscribe", KAFKA_TOPIC)
    .option("startingOffsets", KAFKA_STARTING_OFFSETS)
    .load()
)

kafka_values = kafka_raw.select(
    F.col("key").cast("string").alias("key"),
    F.col("value").cast("string").alias("value"),
    F.col("topic"),
    F.col("partition"),
    F.col("offset"),
    F.col("timestamp").alias("kafka_timestamp"),
)

kafka_values.printSchema()

root
 |-- key: string (nullable = true)
 |-- value: string (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- kafka_timestamp: timestamp (nullable = true)



## 8. Define and parse the Kafka JSON schema

The schema below matches the observed Kafka JSON structure.  
Only events with `page = 'NextSong'` are actual song plays.

In [9]:
event_schema = StructType([
    StructField("ts", LongType(), True),
    StructField("auth", StringType(), True),
    StructField("page", StringType(), True),
    StructField("song", StringType(), True),
    StructField("level", StringType(), True),
    StructField("artist", StringType(), True),
    StructField("gender", StringType(), True),
    StructField("method", StringType(), True),
    StructField("status", IntegerType(), True),
    StructField("userId", StringType(), True),
    StructField("lastName", StringType(), True),
    StructField("location", StringType(), True),
    StructField("track_id", IntegerType(), True),
    StructField("firstName", StringType(), True),
    StructField("sessionId", IntegerType(), True),
    StructField("userAgent", StringType(), True),
    StructField("registration", LongType(), True),
    StructField("itemInSession", IntegerType(), True),
])

events_stream = (
    kafka_values
    .select(
        F.from_json(F.col("value"), event_schema).alias("event"),
        "kafka_timestamp",
    )
    .select("event.*", "kafka_timestamp")
    .filter(F.col("page") == "NextSong")
    .withColumn("user_id", F.col("userId").cast("int"))
    .withColumn("event_time", (F.col("ts") / 1000).cast("timestamp"))
    # Each NextSong event is one played song/title.
    # We sum this column in the aggregations below.
    .withColumn("played_song", F.lit(1))
)

events_stream.printSchema()

root
 |-- ts: long (nullable = true)
 |-- auth: string (nullable = true)
 |-- page: string (nullable = true)
 |-- song: string (nullable = true)
 |-- level: string (nullable = true)
 |-- artist: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- method: string (nullable = true)
 |-- status: integer (nullable = true)
 |-- userId: string (nullable = true)
 |-- lastName: string (nullable = true)
 |-- location: string (nullable = true)
 |-- track_id: integer (nullable = true)
 |-- firstName: string (nullable = true)
 |-- sessionId: integer (nullable = true)
 |-- userAgent: string (nullable = true)
 |-- registration: long (nullable = true)
 |-- itemInSession: integer (nullable = true)
 |-- kafka_timestamp: timestamp (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- played_song: integer (nullable = false)



## 9. Join streaming events with static track data

This is the main join required by the assignment.  
The streaming event contains `track_id`, and the static metadata contains the same key.

In [10]:
enriched_events = (
    events_stream
    .join(track_metadata, on="track_id", how="left")
    .select(
        "event_time",
        "kafka_timestamp",
        "user_id",
        "gender",
        "track_id",
        F.col("song").alias("event_song"),
        F.col("artist").alias("event_artist"),
        "track_name",
        "artist_name",
        "genre_name",
        "album_id",
        "album_name",
        "played_song",
    )
)

enriched_events.printSchema()

root
 |-- event_time: timestamp (nullable = true)
 |-- kafka_timestamp: timestamp (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- track_id: integer (nullable = true)
 |-- event_song: string (nullable = true)
 |-- event_artist: string (nullable = true)
 |-- track_name: string (nullable = true)
 |-- artist_name: string (nullable = true)
 |-- genre_name: string (nullable = true)
 |-- album_id: integer (nullable = true)
 |-- album_name: string (nullable = true)
 |-- played_song: integer (nullable = false)



# Task 1: Create genre-based playlists

Create top-20 playlists for:

- Pop
- Jazz
- Rock

The professor’s requirement is implemented as `sum(played_song)`, where each `NextSong` event contributes `1`.

In [11]:
genre_playlist_counts = (
    enriched_events
    .filter(F.col("genre_name").isin("Pop", "Jazz", "Rock"))
    .groupBy(
        "genre_name",
        "track_id",
        "track_name",
        "artist_name",
    )
    .agg(F.sum("played_song").alias("play_count"))
    .orderBy(F.col("genre_name"), F.col("play_count").desc())
)

genre_playlist_counts.printSchema()

root
 |-- genre_name: string (nullable = true)
 |-- track_id: integer (nullable = true)
 |-- track_name: string (nullable = true)
 |-- artist_name: string (nullable = true)
 |-- play_count: long (nullable = true)



In [12]:
def write_genre_playlists_batch(batch_df: DataFrame, batch_id: int) -> None:
    """
    For each streaming micro-batch, write one top-20 table per required genre.
    """
    for genre in ["Pop", "Jazz", "Rock"]:
        output_table = f"{STUDENT_PREFIX}_playlist_{genre.lower()}"

        playlist_df = (
            batch_df
            .filter(F.col("genre_name") == genre)
            .orderBy(F.col("play_count").desc(), F.col("track_id"))
            .limit(20)
            .select(
                "track_id",
                "track_name",
                "artist_name",
                "play_count",
            )
        )

        write_output_table(playlist_df, output_table, mode="overwrite")
        print(f"Batch {batch_id}: wrote {OUTPUT_SCHEMA}.{output_table}")


genre_playlist_query = (
    genre_playlist_counts
    .writeStream
    .outputMode("complete")
    .foreachBatch(write_genre_playlists_batch)
    .option("checkpointLocation", prepare_checkpoint("genre_playlists"))
    .start()
)

time.sleep(PROCESSING_SECONDS)

genre_playlist_query.stop()
print("Genre playlist query stopped:", not genre_playlist_query.isActive)

26/06/22 22:29:31 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/06/22 22:29:32 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


Batch 0: wrote streaming_data.ds25m028_playlist_pop
Batch 0: wrote streaming_data.ds25m028_playlist_jazz
Batch 0: wrote streaming_data.ds25m028_playlist_rock


Batch 1: wrote streaming_data.ds25m028_playlist_pop
Batch 1: wrote streaming_data.ds25m028_playlist_jazz
Batch 1: wrote streaming_data.ds25m028_playlist_rock


Batch 2: wrote streaming_data.ds25m028_playlist_pop
Batch 2: wrote streaming_data.ds25m028_playlist_jazz
Batch 2: wrote streaming_data.ds25m028_playlist_rock


Genre playlist query stopped: True


# Task 2: Create gender-specific playlists

Create:

- top 20 tracks for male listeners
- top 20 tracks for female listeners
- one gender + genre playlist: male + Rock

Again, play counts are calculated by summing `played_song`.

In [13]:
gender_playlist_counts = (
    enriched_events
    .filter(F.col("gender").isin("M", "F"))
    .groupBy(
        "gender",
        "genre_name",
        "track_id",
        "track_name",
        "artist_name",
    )
    .agg(F.sum("played_song").alias("play_count"))
    .orderBy(F.col("gender"), F.col("play_count").desc())
)

gender_playlist_counts.printSchema()

root
 |-- gender: string (nullable = true)
 |-- genre_name: string (nullable = true)
 |-- track_id: integer (nullable = true)
 |-- track_name: string (nullable = true)
 |-- artist_name: string (nullable = true)
 |-- play_count: long (nullable = true)



In [14]:
def write_gender_playlists_batch(batch_df: DataFrame, batch_id: int) -> None:
    """
    For each streaming micro-batch, write the required gender-based playlists.
    """
    playlists = [
        ("male", F.col("gender") == "M"),
        ("female", F.col("gender") == "F"),
        ("male_rock", (F.col("gender") == "M") & (F.col("genre_name") == "Rock")),
    ]

    for playlist_name, condition in playlists:
        output_table = f"{STUDENT_PREFIX}_playlist_{playlist_name}"

        playlist_df = (
            batch_df
            .filter(condition)
            .orderBy(F.col("play_count").desc(), F.col("track_id"))
            .limit(20)
            .select(
                "track_id",
                "track_name",
                "artist_name",
                "play_count",
            )
        )

        write_output_table(playlist_df, output_table, mode="overwrite")
        print(f"Batch {batch_id}: wrote {OUTPUT_SCHEMA}.{output_table}")


gender_playlist_query = (
    gender_playlist_counts
    .writeStream
    .outputMode("complete")
    .foreachBatch(write_gender_playlists_batch)
    .option("checkpointLocation", prepare_checkpoint("gender_playlists"))
    .start()
)

time.sleep(PROCESSING_SECONDS)

gender_playlist_query.stop()
print("Gender playlist query stopped:", not gender_playlist_query.isActive)

26/06/22 22:30:12 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/06/22 22:30:12 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


Batch 0: wrote streaming_data.ds25m028_playlist_male
Batch 0: wrote streaming_data.ds25m028_playlist_female
Batch 0: wrote streaming_data.ds25m028_playlist_male_rock


Batch 1: wrote streaming_data.ds25m028_playlist_male
Batch 1: wrote streaming_data.ds25m028_playlist_female
Batch 1: wrote streaming_data.ds25m028_playlist_male_rock


Batch 2: wrote streaming_data.ds25m028_playlist_male
Batch 2: wrote streaming_data.ds25m028_playlist_female
Batch 2: wrote streaming_data.ds25m028_playlist_male_rock


Batch 3: wrote streaming_data.ds25m028_playlist_male
Batch 3: wrote streaming_data.ds25m028_playlist_female
Batch 3: wrote streaming_data.ds25m028_playlist_male_rock


Gender playlist query stopped: True


26/06/22 22:30:43 WARN Shell: Interrupted while joining on: Thread[Thread-999,5,main]
java.lang.InterruptedException
	at java.base/java.lang.Object.wait(Native Method)
	at java.base/java.lang.Thread.join(Thread.java:1313)
	at java.base/java.lang.Thread.join(Thread.java:1381)
	at org.apache.hadoop.util.Shell.joinThread(Shell.java:1042)
	at org.apache.hadoop.util.Shell.runCommand(Shell.java:1002)
	at org.apache.hadoop.util.Shell.run(Shell.java:900)
	at org.apache.hadoop.util.Shell$ShellCommandExecutor.execute(Shell.java:1212)
	at org.apache.hadoop.util.Shell.execCommand(Shell.java:1306)
	at org.apache.hadoop.util.Shell.execCommand(Shell.java:1288)
	at org.apache.hadoop.fs.FileUtil.readLink(FileUtil.java:212)
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileLinkStatusInternal(RawLocalFileSystem.java:1113)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1102)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatus(RawLocalFile

26/06/22 22:30:43 WARN TaskSetManager: Lost task 1.0 in stage 369.0 (TID 326) (46b366ee-64cf-4259-bb44-2ddc3ffcdf08.internal.cloudapp.net executor driver): TaskKilled (Stage cancelled: Job 39 cancelled part of cancelled job group 7deac619-0592-48d2-9a7d-9e80e09d9853)
26/06/22 22:30:43 WARN TaskSetManager: Lost task 0.0 in stage 369.0 (TID 325) (46b366ee-64cf-4259-bb44-2ddc3ffcdf08.internal.cloudapp.net executor driver): TaskKilled (Stage cancelled: Job 39 cancelled part of cancelled job group 7deac619-0592-48d2-9a7d-9e80e09d9853)


# Task 3: Create top 20 albums

Aggregate the top 20 albums based on the number of played tracks from each album.

Required columns:

- `album_id`
- `album_name`
- `artist_name`

The notebook also writes `play_count` for transparency.

In [15]:
album_counts = (
    enriched_events
    .groupBy(
        "album_id",
        "album_name",
        "artist_name",
    )
    .agg(F.sum("played_song").alias("play_count"))
    .orderBy(F.col("play_count").desc())
)

album_counts.printSchema()

root
 |-- album_id: integer (nullable = true)
 |-- album_name: string (nullable = true)
 |-- artist_name: string (nullable = true)
 |-- play_count: long (nullable = true)



In [16]:
def write_top_albums_batch(batch_df: DataFrame, batch_id: int) -> None:
    """
    For each streaming micro-batch, write the current top 20 albums.
    """
    output_table = f"{STUDENT_PREFIX}_top_20_albums"

    top_albums_df = (
        batch_df
        .orderBy(F.col("play_count").desc(), F.col("album_id"))
        .limit(20)
        .select(
            "album_id",
            "album_name",
            "artist_name",
            "play_count",
        )
    )

    write_output_table(top_albums_df, output_table, mode="overwrite")
    print(f"Batch {batch_id}: wrote {OUTPUT_SCHEMA}.{output_table}")


top_albums_query = (
    album_counts
    .writeStream
    .outputMode("complete")
    .foreachBatch(write_top_albums_batch)
    .option("checkpointLocation", prepare_checkpoint("top_albums"))
    .start()
)

time.sleep(PROCESSING_SECONDS)

top_albums_query.stop()
print("Top albums query stopped:", not top_albums_query.isActive)

26/06/22 22:30:55 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/06/22 22:30:55 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


Batch 0: wrote streaming_data.ds25m028_top_20_albums


Batch 1: wrote streaming_data.ds25m028_top_20_albums


Batch 2: wrote streaming_data.ds25m028_top_20_albums


Batch 3: wrote streaming_data.ds25m028_top_20_albums


Batch 4: wrote streaming_data.ds25m028_top_20_albums


Batch 5: wrote streaming_data.ds25m028_top_20_albums
Top albums query stopped: True


## 10. Final verification

This section checks that the required PostgreSQL tables exist and displays their contents.

In [17]:
expected_tables = [
    f"{STUDENT_PREFIX}_playlist_pop",
    f"{STUDENT_PREFIX}_playlist_jazz",
    f"{STUDENT_PREFIX}_playlist_rock",
    f"{STUDENT_PREFIX}_playlist_male",
    f"{STUDENT_PREFIX}_playlist_female",
    f"{STUDENT_PREFIX}_playlist_male_rock",
    f"{STUDENT_PREFIX}_top_20_albums",
]

created_tables = read_jdbc_query(f"""
SELECT table_schema, table_name
FROM information_schema.tables
WHERE table_schema = '{OUTPUT_SCHEMA}'
  AND table_name LIKE '{STUDENT_PREFIX}%'
ORDER BY table_name
""")

created_tables.show(100, truncate=False)

print("Expected output tables:")
for table_name in expected_tables:
    print(f"- {OUTPUT_SCHEMA}.{table_name}")

+--------------+---------------------------+
|table_schema  |table_name                 |
+--------------+---------------------------+
|streaming_data|ds25m028_debug_pop_top100  |
|streaming_data|ds25m028_playlist_female   |
|streaming_data|ds25m028_playlist_jazz     |
|streaming_data|ds25m028_playlist_male     |
|streaming_data|ds25m028_playlist_male_rock|
|streaming_data|ds25m028_playlist_pop      |
|streaming_data|ds25m028_playlist_rock     |
|streaming_data|ds25m028_top20_albums      |
|streaming_data|ds25m028_top_20_albums     |
+--------------+---------------------------+

Expected output tables:
- streaming_data.ds25m028_playlist_pop
- streaming_data.ds25m028_playlist_jazz
- streaming_data.ds25m028_playlist_rock
- streaming_data.ds25m028_playlist_male
- streaming_data.ds25m028_playlist_female
- streaming_data.ds25m028_playlist_male_rock
- streaming_data.ds25m028_top_20_albums


In [18]:
for table_name in expected_tables:
    print(f"\n{OUTPUT_SCHEMA}.{table_name}")
    check_df = read_jdbc_query(f"""
        SELECT *
        FROM {OUTPUT_SCHEMA}.{table_name}
        LIMIT 20
    """)
    check_df.show(20, truncate=False)


streaming_data.ds25m028_playlist_pop
+--------+----------+-----------+----------+
|track_id|track_name|artist_name|play_count|
+--------+----------+-----------+----------+
+--------+----------+-----------+----------+


streaming_data.ds25m028_playlist_jazz
+--------+--------------------------------------------+--------------------+----------+
|track_id|track_name                                  |artist_name         |play_count|
+--------+--------------------------------------------+--------------------+----------+
|64      |Garota De Ipanema                           |Antônio Carlos Jobim|3         |
|65      |Samba De Uma Nota Só (One Note Samba)       |Antônio Carlos Jobim|3         |
|68      |Fotografia                                  |Antônio Carlos Jobim|3         |
|69      |Dindi (Dindi)                               |Antônio Carlos Jobim|3         |
|70      |Se Todos Fossem Iguais A Você (Instrumental)|Antônio Carlos Jobim|3         |
|71      |Falando De Amor             